In [5]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image

# ==============================================================================
# CONFIGURATION
# ==============================================================================
IMAGES_DIR    = "/chemin/vers/flickr30k_images"  # modifier ce chemin
BATCH_SIZE    = 32
NUM_EPOCHS    = 10
LEARNING_RATE = 0.001
IMAGE_SIZE    = 224
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device utilise : {DEVICE}")

# ==============================================================================
# ETAPE 1 : DATASET
# Charge les images du dossier, le label de chaque image = son index
# ==============================================================================
class FlickrDataset(Dataset):
    def __init__(self, images_dir, transform=None):
        self.transform   = transform
        self.image_paths = []

        # Recupere tous les fichiers images du dossier
        for fname in sorted(os.listdir(images_dir)):
            if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                self.image_paths.append(os.path.join(images_dir, fname))

        print(f"Nombre d'images chargees : {len(self.image_paths)}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Ouvre l'image et applique les transformations
        image = Image.open(self.image_paths[idx]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(idx, dtype=torch.long)
        return image, label


# ==============================================================================
# ETAPE 2 : TRANSFORMATIONS
# Prepare les images pour ResNet (redimensionnement + normalisation ImageNet)
# ==============================================================================
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),       # augmentation : miroir aleatoire
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225])
])


# ==============================================================================
# ETAPE 3 : CHARGEMENT DES DONNEES
# Split 80% train / 20% validation
# ==============================================================================
train_dataset = FlickrDataset(IMAGES_DIR, transform=train_transform)
val_dataset   = FlickrDataset(IMAGES_DIR, transform=val_transform)

NUM_CLASSES = len(train_dataset)

val_size   = int(0.2 * NUM_CLASSES)
train_size = NUM_CLASSES - val_size

# Meme indices pour les deux datasets (train et val)
train_indices, val_indices = random_split(range(NUM_CLASSES), [train_size, val_size])

train_loader = DataLoader(
    torch.utils.data.Subset(train_dataset, train_indices),
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    torch.utils.data.Subset(val_dataset, val_indices),
    batch_size=BATCH_SIZE,
    shuffle=False
)

print(f"Train : {train_size} images")
print(f"Val   : {val_size} images")


# ==============================================================================
# ETAPE 4 : MODELE ResNet-50
# On charge ResNet-50 pre-entraine puis on remplace sa derniere couche
# ==============================================================================
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

# Gel du backbone : les poids ne seront pas modifies pendant l'entrainement
for param in model.parameters():
    param.requires_grad = False

# Seule la nouvelle couche FC sera entrainee
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

model = model.to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Parametres entrainables : {trainable:,} / {total:,}")


# ==============================================================================
# ETAPE 5 : LOSS ET OPTIMISEUR
# ==============================================================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=LEARNING_RATE)


# ==============================================================================
# ETAPE 6 : ENTRAINEMENT
# ==============================================================================
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct = 0, 0

    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct    += (outputs.argmax(1) == labels).sum().item()

    avg_loss = total_loss / len(loader)
    accuracy = correct / len(loader.dataset) * 100
    return avg_loss, accuracy


# ==============================================================================
# ETAPE 7 : VALIDATION
# ==============================================================================
def validate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0, 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            loss    = criterion(outputs, labels)

            total_loss += loss.item()
            correct    += (outputs.argmax(1) == labels).sum().item()

    avg_loss = total_loss / len(loader)
    accuracy = correct / len(loader.dataset) * 100
    return avg_loss, accuracy


# ==============================================================================
# ETAPE 8 : BOUCLE PRINCIPALE
# ==============================================================================
best_val_acc = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train(model, train_loader, optimizer, criterion)
    val_loss,   val_acc   = validate(model, val_loader, criterion)

    print(f"Epoch {epoch}/{NUM_EPOCHS} "
          f"| Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% "
          f"| Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}%")

    # Sauvegarde du meilleur modele
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "resnet50_best.pth")
        print(f"  Meilleur modele sauvegarde (val acc : {best_val_acc:.2f}%)")

print("Entrainement termine.")


# ==============================================================================
# ETAPE 9 : EXTRACTION DES FEATURES VISUELLES
# Retire la couche FC pour obtenir les vecteurs de features 2048-D
# ==============================================================================
def extract_features(model, loader):
    model.eval()

    # Backbone sans la derniere couche FC
    backbone = nn.Sequential(*list(model.children())[:-1]).to(DEVICE)

    all_features = []
    all_labels   = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            feats  = backbone(images)           # (batch, 2048, 1, 1)
            feats  = feats.flatten(start_dim=1) # (batch, 2048)
            all_features.append(feats.cpu())
            all_labels.append(labels)

    features = torch.cat(all_features)
    labels   = torch.cat(all_labels)
    return features, labels


features, labels = extract_features(model, train_loader)
torch.save({"features": features, "labels": labels}, "flickr30k_features.pth")
print(f"Features extraites et sauvegardees : {features.shape}")

Device utilise : cpu


FileNotFoundError: [Errno 2] No such file or directory: '/chemin/vers/flickr30k_images'